# 04 — Logit Lens & Direct Logit Attribution

Because the residual stream is purely additive (every layer just adds to
it), you can take the residual stream *at any layer* and run it through the
final `LayerNorm` + unembedding (`W_U`) to see what the model "would have
predicted" if it stopped there. This is the **logit lens**.

Going further, since the final logits are a *sum* of each component's
individual contribution (each attention head, each MLP), you can measure
each component's **direct logit attribution (DLA)**: how much did this
specific head push toward the correct answer, independent of everything
else?


## Shorthand in this notebook

- **DLA** — **d**irect **l**ogit **a**ttribution: how much a single
  component (one head, or the MLP) contributed to one specific output
  logit, in isolation from every other component. "Direct" distinguishes it
  from a component's *total* effect, which could also work indirectly by
  feeding into a later layer — DLA only measures the straight-line
  contribution via `W_U`.
- **`accumulated_resid`** — a cache method that returns the residual stream
  summed up through each layer *in turn* (layer 0 alone, then layers 0+1,
  then 0+1+2, ...), rather than one final sum. This is what makes "what
  would the model predict if it stopped here?" answerable per-layer.
- **`incl_mid=True`** — "include mid": also return the halfway point inside
  each layer (`resid_mid`, i.e. after attention but before the MLP has run),
  not just the point after the full layer (`resid_post`). Gives twice as
  many points on the logit-lens curve.
- **`apply_ln_to_stack`** — applies the model's final LayerNorm (`ln`) to a
  whole stack (`stack` = a batch of residual-stream snapshots bundled into
  one extra tensor dimension) of vectors at once. Necessary because
  LayerNorm's scale depends on the specific vector — you can't just skip it
  and multiply by `W_U` directly, or the logit-lens numbers come out wrong.
- **`stack_head_results`** — like `accumulated_resid`, but slices the
  residual stream contribution *per head* instead of *per layer*: one
  vector per (layer, head) pair, all bundled ("stacked") together.
- **`n_points`** (in the shape comment `[n_points, batch, pos, d_vocab]`) —
  however many residual-stream snapshots got stacked (number of layers ×2
  from `incl_mid`, or number of heads, depending on which stacking function
  was used).


In [1]:
import torch
from transformer_lens import HookedTransformer, utils
import plotly.express as px

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_68341/3108316550.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17933.70it/s]

Loaded pretrained model gpt2 into HookedTransformer


In [2]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
str_tokens = model.to_str_tokens(text)
logits, cache = model.run_with_cache(tokens)

correct_token = model.to_single_token(" Paris")
print("target token id:", correct_token)


target token id: 6342


## Logit lens: prediction confidence by layer

`cache.accumulated_resid` gives you the residual stream summed up to (and
including) each layer. Apply the model's final layer norm and unembedding to
each one.


In [3]:
accumulated = cache.accumulated_resid(layer=-1, incl_mid=True, return_labels=True)
resid_stack, labels = accumulated

# apply_ln_to_stack handles the final LayerNorm correctly (it's per-position, not global)
scaled = cache.apply_ln_to_stack(resid_stack, layer=-1)
logit_lens_logits = scaled @ model.W_U  # [n_points, batch, pos, d_vocab]

last_pos_logits = logit_lens_logits[:, 0, -1, :]  # last token position, only batch item
target_logit = last_pos_logits[:, correct_token]
probs = last_pos_logits.softmax(dim=-1)[:, correct_token]

px.line(x=labels, y=probs.cpu(), markers=True,
        labels=dict(x="layer (accumulated up to here)", y="P(' Paris')"),
        title="Logit lens: how confidence in the correct answer builds up").show()


## Direct logit attribution per head

Instead of accumulating layers, decompose the *final* residual stream into
each individual component's contribution, then project each one through
`W_U` toward the correct token's direction. This tells you which specific
heads/MLPs are "voting" for the right answer.


In [4]:
per_head_resid, head_labels = cache.stack_head_results(layer=-1, return_labels=True)
scaled_heads = cache.apply_ln_to_stack(per_head_resid, layer=-1)

# direction in residual-stream space that increases the correct token's logit
correct_direction = model.W_U[:, correct_token]  # [d_model]

dla = scaled_heads[:, 0, -1, :] @ correct_direction  # [n_heads_total]
dla = dla.reshape(model.cfg.n_layers, model.cfg.n_heads)

px.imshow(dla.cpu(), labels=dict(x="head", y="layer"),
          title="Direct logit attribution toward ' Paris'",
          color_continuous_scale="RdBu", color_continuous_midpoint=0).show()


## Next

Logit lens and DLA tell you *which* components matter for a given output,
but only correlationally. `05_activation_patching.ipynb` gives you the
*causal* version: swap a component's activation between two runs and measure
what actually changes.
